In [ ]:
%load_ext autoreload
%autoreload 2
    
import safetensors.torch
from models.FEMBA import FEMBA, FembaEncoder

In [ ]:
from huggingface_hub import snapshot_download

# downloads all task folders (TUAB/TUAR/TUSL) and safetensors into ./checkpoints/FEMBA
snapshot_download(repo_id="thorir/FEMBA", repo_type="model", local_dir="checkpoints/FEMBA")

In [ ]:
def load_model_from_safetensors(safetensors_path, device="cpu"):
    weights = safetensors.torch.load_file(safetensors_path)
    model_full = FEMBA(num_classes=2)
    model_encoder = FembaEncoder()
    for model in model_full, model_encoder:
        model.load_state_dict(weights, strict=False)
        model.eval()
        model.to(device)
    
    return model_full, model_encoder

device="cuda"
model_f, model_e = load_model_from_safetensors("checkpoints/FEMBA/TUAR/FEMBA_large.safetensors", device)

In [ ]:
import torch

batch_size = 4
seq_length = 1280
num_channels = 22

# Create random EEG data (batch_size, channels, seq_length)
x = torch.randn(batch_size, num_channels, seq_length).to(device)

# Create a mask (same shape as input, boolean)
# Mask indicates which positions should be masked (set to True for masked positions)
mask = torch.zeros(batch_size, num_channels, seq_length, dtype=torch.bool).to(device)
# Mask some random positions (e.g., mask 10% of the input)
mask[:, :, torch.randint(0, seq_length, (int(0.1 * seq_length),))] = True

# Forward pass
with torch.no_grad():  # Disable gradient computation for inference
    output, original = model_f(x, mask)
    embedding = model_e(x, mask)

print(f"Input shape: {x.shape}")
print(f"Mask shape: {mask.shape}")
print(f"Output shape: {output.shape}")  # For classification: (batch_size, num_classes) = (4, 2)
print(f"Embedding shape: {embedding.shape}") # (4, 80, 869)
print(f"Original shape: {original.shape}")  # (batch_size, num_channels, seq_length) = (4, 22, 1280)

# preprocess and run short EDF file

In [ ]:
import mne
import numpy as np
import torch
from make_datasets.process_raw_eeg import CH_ORDER_STANDARD, TCP_BIPOLAR_MONTAGE

In [ ]:
def make_bipolar(raw):
    channels = raw.ch_names
    data = raw.get_data(units='uV')
    new_data = []
    new_channels = []
    for new_ch_name, (ch1, ch2) in TCP_BIPOLAR_MONTAGE.items():
        if ch1 in channels and ch2 in channels:
            new_data.append(data[channels.index(ch1)] - data[channels.index(ch2)])
            new_channels.append(new_ch_name)
    return np.array(new_data), new_channels

edf_path = 'datasets/test_from_20s.edf'
raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

In [ ]:
from tqdm import tqdm

# Rename channels to match standard (replace -A1/-A2 with -REF)
raw.rename_channels(lambda x: x.replace('-A1', '-REF').replace('-A2', '-REF'))

# dummy REF
missing_channels = ['EEG A1-REF', 'EEG A2-REF'] # not in CH_ORDER_STANDARD
for ch in missing_channels:
    if ch not in raw.ch_names:
        # Create a dummy channel with zeros
        dummy_data = np.zeros((1, raw.n_times))
        dummy_info = mne.create_info(ch_names=[ch], sfreq=raw.info['sfreq'], ch_types='eeg')
        dummy_raw = mne.io.RawArray(dummy_data, dummy_info, verbose=False)
        raw.add_channels([dummy_raw], force_update_info=True)

raw.pick_channels(CH_ORDER_STANDARD, ordered=True)

# Filter and resample
raw.filter(l_freq=0.1, h_freq=75.0, verbose=False)
raw.notch_filter(60, verbose=False)
raw.resample(256, npad="auto", verbose=False)

# Convert to bipolar
channeled_data, channeled_channels = make_bipolar(raw)
n_channels, n_times = channeled_data.shape
window_size_samples = 1280  # 5 seconds at 256 Hz

# Split into epochs
epochs = []
for i in tqdm(range(n_times // window_size_samples), "epochs"):
    segment = channeled_data[:, i * window_size_samples: (i + 1) * window_size_samples]
    epochs.append(segment)

In [ ]:
embeddings = []
model_e.eval()
with torch.no_grad():
    for epoch in tqdm(epochs):
        x = torch.FloatTensor(epoch).unsqueeze(0)  # (1, 22, 1280)
        mask = torch.zeros_like(x, dtype=torch.bool)  # No masking for inference
        emb = model_e(x, mask)
        embeddings.append(emb.cpu().numpy())